# 4-6. Candidate endpoint 운영 관측 실습

이 notebook은 후보 모델 endpoint의 로그와 metric을 배포 확인 확인 결과로 정리하는 실습입니다. 모델 평가 파일만으로는 Argo CD/KServe 배포 이후 입력 오류, 지연 시간, prediction distribution 변화를 설명할 수 없으므로, structured log와 dashboard artifact를 함께 확인합니다.


## 1. 실행 환경과 관측 확인 결과 범위

### 1-1. Notebook helper와 prepared artifact 범위 확인

이 셀에서는 후보 모델 endpoint 관측 확인 결과를 어디서 읽는지 고정하는 것입니다. JupyterLite는 Prometheus, Loki, Grafana Cloud, Argo CD, KServe를 직접 실행하지 않으므로 prepared telemetry를 배포 확인 입력으로 해석합니다.


In [ ]:
from __future__ import annotations

import json

import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
lite = prepared.aiq_lite

for name in runtime.LITE_NAMES:
    globals()[name] = getattr(lite, name)

artifact_contract = pd.DataFrame(
    [
        {"artifact": "normal_events", "path": "artifacts/logs/chapter_04_normal_events.jsonl", "role": "operational_baseline structured log"},
        {"artifact": "anomaly_events", "path": "artifacts/logs/chapter_04_anomaly_events.jsonl", "role": "current/anomaly structured log"},
        {"artifact": "prometheus_metrics", "path": "artifacts/metrics/chapter_04_anomaly.prom", "role": "metric name/value 확인 결과"},
        {"artifact": "grafana_overview", "path": "artifacts/grafana/ai_quality_overview_dashboard.json", "role": "dashboard panel/query inspection"},
        {"artifact": "grafana_payload_preview", "path": "artifacts/grafana/grafana_cloud_payload_preview.json", "role": "Grafana Cloud 전송 전 payload preview"},
    ]
)
runtime_contract = pd.DataFrame(
    [
        {"item": "package_module", "value": lite.__name__, "qa_use": "Lite 또는 local helper 기준을 확인합니다."},
        {"item": "notebook_scope", "value": "structured_log_and_metric_inspection", "qa_use": "Grafana Cloud 전송은 수행하지 않습니다."},
        {"item": "baseline_layer", "value": "operational_baseline", "qa_use": "4장은 운영 기준선과 current/anomaly를 비교합니다."},
    ]
)

display(runtime_contract)
display(artifact_contract)

이 출력에서 확인할 핵심은 prepared artifact와 live monitoring을 구분하는 것입니다. Notebook에서 dashboard JSON을 읽었다고 해서 Grafana Cloud 전송이나 live KServe endpoint 수집을 검증한 것은 아니며, 보고서에는 artifact inspection이라고 적어야 합니다.

## 2. structured log 계약 확인

### 2-1. champion 기준선과 candidate event sample 로드

이 셀에서는 candidate 배포 확인에 필요한 로그 필드가 같은 구조로 남는지 확인하는 것입니다. field 구조가 다르면 metric 변화 이전에 observability contract 문제부터 해결해야 합니다.


In [ ]:
def read_jsonl_artifact(path: str, max_events: int | None = 120) -> list[dict[str, object]]:
    text = runtime.read_text_artifact(path)
    events = [json.loads(line) for line in text.splitlines() if line.strip()]
    return events[:max_events] if max_events is not None else events


normal_events = read_jsonl_artifact("artifacts/logs/chapter_04_normal_events.jsonl")
current_events = read_jsonl_artifact("artifacts/logs/chapter_04_anomaly_events.jsonl")
normal_log = pd.DataFrame(normal_events)
current_log = pd.DataFrame(current_events)

required_log_fields = [
    "timestamp",
    "request_id",
    "trace_id",
    "model_version",
    "score",
    "threshold",
    "prediction",
    "latency_ms",
    "status_code",
    "validation_failure",
    "client_id",
    "source_system",
    "failed_field",
    "owner",
]
log_data_brief = pd.DataFrame(
    [
        {"dataset": "normal_events", "source": "chapter_04_normal_events.jsonl", "row_grain": "one prediction event", "row_count": len(normal_log), "column_count": len(normal_log.columns)},
        {"dataset": "current_events", "source": "chapter_04_anomaly_events.jsonl", "row_grain": "one prediction event", "row_count": len(current_log), "column_count": len(current_log.columns)},
    ]
)
field_contract = pd.DataFrame(
    [
        {
            "field": field,
            "normal_present": field in normal_log.columns,
            "current_present": field in current_log.columns,
            "qa_role": "trace" if field in ["request_id", "trace_id"] else "quality_signal",
            "qa_status": "pass" if field in normal_log.columns and field in current_log.columns else "보류",
        }
        for field in required_log_fields
    ]
)

display(log_data_brief)
display(field_contract)
display(current_log.loc[:, [field for field in required_log_fields if field in current_log.columns]].head())

이 출력에서 확인할 핵심은 정상 기준선과 이상 상태 로그가 같은 필드 구조를 가진다는 점입니다. 필드 구조가 다르면 metric 변화 이전에 로그 계약 문제부터 해결해야 합니다.

로그 field는 단순 디버깅 정보가 아닙니다. 각 field는 운영 질문에 답하기 위한 증거이므로, 누락된 field가 있으면 owner와 재평가 조건을 남겨야 합니다.

### 2-2. field별 QA 질문과 누락 리스크 정리

이 셀에서는 로그 field가 어떤 QA 질문에 답하는지 명확히 하는 것입니다. 필드 이름만 보면 왜 필요한지 잊기 쉽기 때문에, 질문과 누락 리스크를 함께 둡니다.

`request_id`가 없으면 특정 요청을 다시 찾기 어렵고, `threshold`가 없으면 prediction 변화가 설정 변경 때문인지 알기 어렵습니다. `failed_field`와 `owner`가 없으면 validation failure가 증가해도 조치가 “확인 필요”에서 멈춥니다.

이 표는 이후 request 추적과 dashboard panel 해석의 기준표입니다.

In [ ]:
field_meaning = pd.DataFrame(
    [
        {"field": "request_id", "qa_question": "어떤 요청에서 문제가 발생했는가", "missing_risk": "대표 요청 재현이 어렵습니다."},
        {"field": "trace_id", "qa_question": "같은 처리 흐름의 로그를 묶을 수 있는가", "missing_risk": "API, 검증, 추론 단계를 연결하기 어렵습니다."},
        {"field": "model_version", "qa_question": "어떤 모델이 응답했는가", "missing_risk": "배포 변경과 품질 변화를 연결하기 어렵습니다."},
        {"field": "score", "qa_question": "threshold 적용 전 모델 출력은 얼마인가", "missing_risk": "prediction 변화 원인을 좁히기 어렵습니다."},
        {"field": "threshold", "qa_question": "어떤 기준으로 class가 결정되었는가", "missing_risk": "설정 변경과 score 변화를 분리하기 어렵습니다."},
        {"field": "prediction", "qa_question": "최종 예측 class는 무엇인가", "missing_risk": "prediction distribution을 볼 수 없습니다."},
        {"field": "validation_failure", "qa_question": "입력 계약 위반이 늘었는가", "missing_risk": "API 입력 문제와 모델 응답 문제를 섞게 됩니다."},
        {"field": "failed_field", "qa_question": "어떤 입력 필드가 실패했는가", "missing_risk": "client/upstream 조치가 구체화되지 않습니다."},
        {"field": "owner", "qa_question": "누가 다음 확인을 맡는가", "missing_risk": "보고서가 action 없이 끝납니다."},
    ]
)

display(field_meaning)

이 표에서 확인할 핵심은 로그 설계가 QA 테스트 기준이라는 점입니다. “로그가 있다”가 아니라 “필요한 질문에 답할 수 있는 필드가 있다”가 배포 확인 기준입니다.

## 3. request_id로 대표 요청 추적

### 3-1. 이상 상태 대표 요청과 trace 묶음 확인

이 셀에서는 aggregate metric에서 대표 요청 로그로 내려갈 수 있는지 확인하는 것입니다. dashboard에서 오류율이나 high_risk 비율이 증가해도, 특정 요청을 찾지 못하면 원인 후보를 좁히기 어렵습니다.

먼저 validation failure가 있는 대표 요청 하나를 선택합니다. 그다음 같은 `trace_id`에 묶인 로그를 확인해, 하나의 처리 흐름 안에서 어떤 요청들이 함께 관측되는지 봅니다.

이 단계의 목적은 전체 원인 확정이 아니라, 개별 요청 재현과 owner 연결이 가능한지 확인하는 것입니다.

In [ ]:
failed_events = current_log.loc[current_log["validation_failure"].astype(bool)].copy()
target_request_id = str(failed_events.iloc[0]["request_id"])
target_trace_id = str(failed_events.iloc[0]["trace_id"])

target_request = current_log.loc[current_log["request_id"].eq(target_request_id), required_log_fields]
trace_group = current_log.loc[current_log["trace_id"].eq(target_trace_id), required_log_fields].head(10)
trace_summary = pd.DataFrame(
    [
        {"check": "target_request_id", "observed": target_request_id, "qa_use": "대표 실패 요청을 재현 가능한 식별자로 남깁니다."},
        {"check": "target_trace_id", "observed": target_trace_id, "qa_use": "같은 처리 흐름의 이벤트를 묶습니다."},
        {"check": "trace_event_count", "observed": int(current_log["trace_id"].eq(target_trace_id).sum()), "qa_use": "trace로 확장 가능한 이벤트 수입니다."},
    ]
)

display(trace_summary)
display(target_request)
display(trace_group)

이 출력에서 확인할 핵심은 대표 요청이 `failed_field`, `client_id`, `source_system`, `owner`까지 포함한다는 점입니다. 이 정보가 있어야 “검증 실패 증가”를 조치 가능한 업무로 바꿀 수 있습니다.

대표 요청 하나만으로 전체 장애 원인을 확정하지 않습니다. 하지만 이 요청은 aggregate metric에서 내려가 확인할 수 있는 확인 sample이며, 보고서에는 최소 한 개의 request_id를 남기는 것이 좋습니다.

### 3-2. validation failure를 owner와 failed field로 집계

이 셀에서는 검증 실패를 단순 오류 수가 아니라 owner가 있는 조치 항목으로 바꾸는 것입니다. validation failure가 증가하면 모델 지표보다 API 입력 품질을 먼저 확인해야 합니다.

검증 실패 요청은 정상 prediction 분석과 분리해야 합니다. 잘못된 입력이 예측까지 가지 않았다면 score distribution에 섞어 모델 품질로 해석하는 것은 부적절합니다.

아래 집계는 어떤 client/source와 field에서 실패가 반복되는지 보여줍니다.

In [ ]:
validation_failure_summary = (
    current_log.loc[current_log["validation_failure"].astype(bool)]
    .groupby(["client_id", "source_system", "failed_field", "owner"], dropna=False)
    .size()
    .rename("failure_count")
    .reset_index()
    .sort_values("failure_count", ascending=False)
)
validation_failure_check = pd.DataFrame(
    [
        {"check": "validation_failure_count", "observed": int(current_log["validation_failure"].sum()), "qa_interpretation": "입력 계약 위반 규모입니다."},
        {"check": "failed_field_count", "observed": current_log.loc[current_log["validation_failure"].astype(bool), "failed_field"].nunique(dropna=True), "qa_interpretation": "반복 실패 필드가 특정되는지 봅니다."},
        {"check": "owner_assigned", "observed": bool(validation_failure_summary["owner"].notna().all()), "qa_interpretation": "조치 owner가 있는지 확인합니다."},
    ]
)

display(validation_failure_check)
display(validation_failure_summary)

이 출력에서 확인할 핵심은 검증 실패가 client 또는 upstream payload 확인으로 이어진다는 점입니다. 모델 자체 문제로 단정하기 전에 입력 계약 위반 증가를 분리해야 합니다.

## 4. metric 변환과 정상/이상 비교

### 4-1. 로그에서 Prometheus형 metric으로 변환

이 셀에서는 로그가 dashboard에 올라갈 수 있는 숫자로 변환되는지 확인하는 것입니다. metric은 개별 요청의 상세 원인을 말해 주지는 않지만, 변화 규모와 방향을 빠르게 보여줍니다.

여기서 보는 핵심 metric은 request total, error total, validation failure total, latency average, score average, high_risk rate입니다. 각각은 다른 원인 후보와 연결됩니다.

먼저 notebook helper로 current snapshot을 만들고 Prometheus text 형태를 확인합니다. 그다음 prepared metric artifact의 주요 line도 함께 읽어, notebook 계산과 artifact inspection을 연결합니다.

In [ ]:
baseline_snapshot = quality_snapshot(normal_events)
current_snapshot = quality_snapshot(current_events)
rendered_prometheus = render_prometheus(current_snapshot)
prepared_prometheus_text = runtime.read_text_artifact("artifacts/metrics/chapter_04_anomaly.prom")

snapshot_table = pd.DataFrame(
    [
        {"metric": key, "baseline": baseline_snapshot.get(key), "current": current_snapshot.get(key)}
        for key in sorted(set(baseline_snapshot) | set(current_snapshot))
    ]
)
prepared_metric_rows = []
for line in prepared_prometheus_text.splitlines():
    if not line or line.startswith("#"):
        continue
    name_and_labels, value = line.rsplit(" ", 1)
    prepared_metric_rows.append({"metric_line": name_and_labels, "value": value})
prepared_metric_table = pd.DataFrame(prepared_metric_rows).head(12)

print(rendered_prometheus)
display(snapshot_table)
display(prepared_metric_table)

이 출력에서 확인할 핵심은 metric 이름과 의미를 보고서에 남길 수 있어야 한다는 점입니다. `ai_quality_high_risk_rate`가 증가했다는 말은 시작점이고, 원인은 로그와 기준선 비교로 다시 내려가야 합니다.

prepared metric artifact에는 dashboard에서 쓰는 추가 metric도 포함됩니다. Notebook에서 모든 운영 쿼리를 실행하지 않더라도, 어떤 metric 이름이 어떤 확인에 쓰이는지는 확인해야 합니다.

### 4-2. 정상 기준선 대비 delta와 원인 후보 분리

이 셀에서는 현재 값 자체보다 기준선 대비 변화량을 보는 것입니다. 운영 관측에서는 baseline이 있어야 현재 상태가 이상인지 확인할 수 있습니다.

오류율, 지연 시간, high_risk 비율, 평균 score가 함께 증가하면 원인을 하나로 단정하지 않습니다. validation failure, 서비스 지연, 입력 분포 변화, prediction shift를 분리해야 합니다.

이 비교는 5장 QA 전략에서 보류/approve 확인의 운영 근거가 됩니다.

In [ ]:
quality_delta = compare_snapshots(baseline_snapshot, current_snapshot)
quality_delta_table = pd.DataFrame(
    [
        {"signal": "error_rate_delta", "observed": quality_delta["error_rate_delta"], "candidate": "api_validation"},
        {"signal": "latency_delta_ms", "observed": quality_delta["latency_delta_ms"], "candidate": "service_latency"},
        {"signal": "high_risk_rate_delta", "observed": quality_delta["high_risk_rate_delta"], "candidate": "prediction_shift"},
        {"signal": "average_score_delta", "observed": quality_delta["average_score_delta"], "candidate": "score_distribution_shift"},
    ]
)
notes_table = pd.DataFrame([{"note": note} for note in quality_delta["notes"]])

display(quality_delta_table)
display(notes_table)

이 출력에서 확인할 핵심은 원인 후보를 분리하는 것입니다. `high_risk` 비율 증가와 latency 증가가 동시에 보이면 모델 문제 하나로 합치지 않고, 입력 분포와 운영 부하를 따로 확인합니다.

## 5. 입력 분포와 score/prediction 변화 연결

### 5-1. serving request feature 변화 확인

이 셀에서는 score와 prediction 변화가 입력 feature 변화와 같은 방향인지 확인하는 것입니다. 운영 로그만 보면 output 변화는 보이지만, 입력 값이 어떻게 달라졌는지 별도로 확인해야 합니다.

`serving_requests_valid.csv`는 운영 기준선 입력이고, `serving_requests_current.csv`는 current batch 입력입니다. 한 row는 serving request sample이며, 모델 평가용 label이 아니라 운영 입력 feature를 확인하는 데이터입니다.

이 단계에서는 feature 평균 변화와 score/prediction 변화의 연결 가능성을 봅니다. drift를 확정하지 않고, 5장으로 넘길 원인 후보를 정리합니다.

In [ ]:
fallback_requests = sample_vital_signs().head(120)
baseline_requests, baseline_request_source = load_csv_or_sample(
    "data/serving_requests_valid.csv",
    fallback_requests,
    nrows=None,
)
current_requests, current_request_source = load_csv_or_sample(
    "data/serving_requests_current.csv",
    fallback_requests,
    nrows=None,
)
request_data_brief = pd.DataFrame(
    [
        {"dataset": "serving_requests_valid", "source": baseline_request_source, "row_grain": "one serving request sample", "row_count": len(baseline_requests)},
        {"dataset": "serving_requests_current", "source": current_request_source, "row_grain": "one serving request sample", "row_count": len(current_requests)},
    ]
)
feature_comparison = compare_input_distribution(baseline_requests, current_requests, FEATURE_COLUMNS)
score_comparison = score_distribution_comparison(normal_events, current_events)
score_comparison_table = pd.DataFrame([score_comparison])

display(request_data_brief)
display(feature_comparison.sort_values("shifted", ascending=False))
display(score_comparison_table)

이 출력에서 확인할 핵심은 입력 변화와 output 변화가 같은 방향인지 보는 것입니다. feature 변화가 있고 high_risk rate도 증가하면 입력 case mix 변화를 강한 원인 후보로 남길 수 있습니다.

하지만 이것만으로 drift를 확정하지 않습니다. 운영 기간, 데이터 수집 경로, 배포 변경, threshold 변경이 함께 검토되어야 하며, 그 확인은 5장 QA 전략으로 이어집니다.

### 5-2. 원인 후보를 owner와 next action으로 정리

이 셀에서는 관측값을 action이 있는 원인 후보로 바꾸는 것입니다. 보고서가 “이상이 있습니다”에서 끝나면 배포 확인에서 쓸 수 없습니다.

`trace_candidates`는 입력 feature 변화, prediction shift, validation failure, latency delta를 각각 다른 owner와 next action으로 분리합니다. helper를 사용하지만, 앞선 셀에서 데이터와 metric 변환 과정을 이미 확인했기 때문에 black box가 아닙니다.

이 표는 5장 최종 승인 조건 확인의 운영 증거로 넘어갑니다.

In [ ]:
candidate_table = trace_candidates(
    feature_comparison=feature_comparison,
    score_comparison=score_comparison,
    quality_report=quality_delta,
    current_events=current_events,
)

display(candidate_table)

이 출력에서 확인할 핵심은 원인 후보가 하나가 아니라는 점입니다. 입력 case mix, API validation, service latency, prediction shift는 각각 다른 owner와 후속 확인이 필요합니다.

## 6. dashboard artifact와 Grafana Cloud payload preview

### 6-1. dashboard panel과 query가 관측 질문에 맞는지 확인

이 셀에서는 dashboard JSON이 운영자가 같은 증거를 반복 확인할 수 있게 구성되었는지 보는 것입니다. dashboard는 예쁜 화면이 아니라 metric과 로그를 반복 조회하는 계약입니다.

Grafana Cloud로 직접 보내는 코드는 로컬 Demo 또는 운영 코드에서 다루는 것이 더 적절합니다. Notebook에서는 panel title, query, datasource, payload label, log field가 충분한지 확인합니다.

이 단계에서도 live Grafana 전송을 검증했다고 쓰면 안 됩니다. prepared JSON artifact inspection으로 범위를 제한합니다.

In [ ]:
overview_dashboard = runtime.read_json_artifact("artifacts/grafana/ai_quality_overview_dashboard.json")
details_dashboard = runtime.read_json_artifact("artifacts/grafana/ai_quality_details_dashboard.json")
payload_preview = runtime.read_json_artifact("artifacts/grafana/grafana_cloud_payload_preview.json")

panel_rows = []
for dashboard in [overview_dashboard, details_dashboard]:
    for panel in dashboard.get("panels", []):
        targets = panel.get("targets", [])
        expressions = [target.get("expr") for target in targets if target.get("expr")]
        panel_rows.append(
            {
                "dashboard": dashboard.get("title"),
                "panel_title": panel.get("title"),
                "panel_type": panel.get("type"),
                "query": expressions[0] if expressions else "not_metric_query",
                "qa_use": panel.get("description", "")[:120],
            }
        )
panel_table = pd.DataFrame(panel_rows)
payload_label_table = pd.DataFrame(
    [{"label": key, "value": value} for key, value in payload_preview.get("labels", {}).items()]
)
payload_log_field_table = pd.DataFrame(
    [
        {"field": field, "present_in_preview": field in payload_preview.get("log_events", [{}])[0]}
        for field in required_log_fields
    ]
)

display(panel_table)
display(payload_label_table)
display(payload_log_field_table)

이 출력에서 확인할 핵심은 dashboard가 `request_id`, `validation_failure`, latency, score, prediction을 반복적으로 찾을 수 있게 하는지입니다. panel이 있어도 query가 불명확하면 운영 해석 근거가 약합니다.

payload preview는 전송 전 점검 자료입니다. 실제 Grafana Cloud 수신, 권한, datasource 연결은 이 notebook이 아니라 로컬 Demo 또는 운영 배포 단계에서 확인합니다.

## 7. QA 해석과 5장 handoff

### 7-1. 운영 관측 결과를 보고 문장으로 정리

마지막 정리는 로그와 metric을 하나의 배포 확인 확인 결과으로 바꾸는 것입니다. 숫자와 dashboard 이름만 나열하면 승인 조건 확인을 방어하기 어렵습니다.

보고 문장은 기준선 대비 변화, 대표 request_id, validation failure owner, prediction/score 변화, dashboard artifact 경계를 함께 담아야 합니다. 이 문장은 5장에서 최종 QA 전략과 배포 확인로 이어집니다.

이 출력은 수강생이 그대로 보고서 초안에 옮길 수 있는 확인 결과입니다.

In [ ]:
observability_packet = pd.DataFrame(
    [
        {
            "확인 결과": "structured_log_contract",
            "observed": f"normal_rows={len(normal_log)}, current_rows={len(current_log)}, required_fields_pass={bool(field_contract['qa_status'].eq('pass').all())}",
            "qa_judgment": "request_id와 품질 필드가 structured log에 남아 있습니다.",
            "owner": "QA/Observability",
            "next_action": "대표 request_id와 trace_id를 보고서에 남깁니다.",
        },
        {
            "확인 결과": "operational_delta",
            "observed": f"error_rate_delta={quality_delta['error_rate_delta']}, latency_delta_ms={quality_delta['latency_delta_ms']}, high_risk_rate_delta={quality_delta['high_risk_rate_delta']}",
            "qa_judgment": "오류율, 지연 시간, prediction 분포 변화를 분리해 봐야 합니다.",
            "owner": "QA/Platform/ML Engineering",
            "next_action": "candidate table의 owner별 확인을 진행합니다.",
        },
        {
            "확인 결과": "validation_failure_trace",
            "observed": f"request_id={target_request_id}, failed_field={failed_events.iloc[0]['failed_field']}, owner={failed_events.iloc[0]['owner']}",
            "qa_judgment": "입력 계약 위반이 조치 가능한 단위로 남았습니다.",
            "owner": str(failed_events.iloc[0]["owner"]),
            "next_action": "client/upstream payload 변경 이력을 확인합니다.",
        },
        {
            "확인 결과": "dashboard_boundary",
            "observed": f"overview_panels={len(overview_dashboard.get('panels', []))}, details_panels={len(details_dashboard.get('panels', []))}",
            "qa_judgment": "prepared dashboard JSON은 inspection 확인 결과이며 live Grafana 수신 증거는 아닙니다.",
            "owner": "QA/MLOps",
            "next_action": "운영 연결 시 datasource와 수신 상태를 별도 검증합니다.",
        },
    ]
)
report_sentence = (
    f"운영 로그 비교에서 error_rate_delta={quality_delta['error_rate_delta']}, "
    f"latency_delta_ms={quality_delta['latency_delta_ms']}, "
    f"high_risk_rate_delta={quality_delta['high_risk_rate_delta']}가 관측되었습니다. "
    f"대표 실패 요청은 request_id={target_request_id}, failed_field={failed_events.iloc[0]['failed_field']}, "
    f"owner={failed_events.iloc[0]['owner']}로 추적됩니다. "
    "따라서 모델 문제로 단정하지 않고 API validation, 입력 case mix, service latency, prediction shift를 분리해 5장 승인 조건 확인으로 넘깁니다."
)

display(observability_packet)
print(report_sentence)

### 7-2. 실패 시 확인 포인트

로그 파일을 읽지 못하면 먼저 `artifacts/logs/chapter_04_normal_events.jsonl`과 `artifacts/logs/chapter_04_anomaly_events.jsonl`이 Lite files 또는 로컬 repo에 복사되었는지 확인합니다. 이 notebook은 report artifact와 같은 비교를 위해 앞 120개 이벤트만 사용합니다. 로그가 없으면 metric과 dashboard 확인도 진행할 수 없습니다.

필수 field가 누락되면 dashboard query보다 로그 계약을 먼저 확인합니다. `request_id`, `trace_id`, `model_version`, `score`, `threshold`, `prediction`, `validation_failure`, `latency_ms`가 없으면 운영 추적이 끊깁니다.

Grafana JSON을 읽었더라도 live Grafana Cloud 전송을 검증한 것은 아닙니다. 전송, datasource 연결, query 실행은 별도 로컬 Demo 또는 운영 환경에서 확인해야 합니다.

metric 변화가 보여도 모델 문제로 바로 결론 내리지 않습니다. validation failure, 입력 feature 변화, latency, model version, threshold를 함께 확인하고 owner별 next action을 남깁니다.